D#%% md
# personal.ipynb

Ноутбук собирает единый реестр персонала из трёх источников:
- **Первоначальный план**
- **План по численности**
- **Факт по численности**

После объединения строится отдельная витрина ресурсов по горизонтам:
- **D** — ресурсы на дату `D`
- **D+1** — ресурсы на дату `D+1`, показанные в строке даты `D`
- **D+7** — ресурсы на дату `D+7`, показанные в строке даты `D`

Важно:
- для расчёта горизонтов используется **сдвиг по календарной дате**, а не `shift()` по строкам;
- `shift()` здесь не подходит, потому что зависит от сортировки и количества строк в группе.


In [80]:

from pathlib import Path

import numpy as np
import pandas as pd


# Словарь русских названий месяцев.
RU_MONTHS = {
    1: "январь",
    2: "февраль",
    3: "март",
    4: "апрель",
    5: "май",
    6: "июнь",
    7: "июль",
    8: "август",
    9: "сентябрь",
    10: "октябрь",
    11: "ноябрь",
    12: "декабрь",
}


# Базовый путь к исходным Excel-файлам.
BASE_PATH = Path("source")


In [81]:

# ------------------------------
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ------------------------------

def _to_datetime_safe(df: pd.DataFrame, columns) -> pd.DataFrame:
    """
    Безопасно преобразует существующие колонки в datetime.
    Если колонки нет, функция её пропускает.
    """
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df



def _expand_dates(start, end):
    """
    Возвращает список дат от start до end включительно.

    Используется для разворота диапазона работы сотрудника
    в ежедневные строки.
    """
    if pd.isna(start) or pd.isna(end):
        return []
    if end < start:
        return []

    return pd.date_range(start=start, end=end, freq="D").date.tolist()


In [82]:

# ------------------------------
# 1. ПЕРВОНАЧАЛЬНЫЙ ПЛАН
# ------------------------------

def transform_initial_plan(df: pd.DataFrame) -> pd.DataFrame:
    """
    Переводит таблицу 'Первоначальный план' в унифицированный формат.

    Логика повторяет Power Query:
    - приводит типы;
    - разворачивает период [Дата.приема.2026] -> [Дата.увольнения.2026]
      в ежедневные строки;
    - переименовывает колонки в общий формат;
    - оставляет только 2026 год.
    """
    df = df.copy()

    # Приводим текстовые колонки к строковому типу.
    text_cols = [
        "Подразделение",
        "Источник",
        "ЦФО",
        "Должность",
        "ФИО ",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # Даты приема и увольнения приводим к типу date.
    if "Дата.приема.2026" in df.columns:
        df["Дата.приема.2026"] = pd.to_datetime(
            df["Дата.приема.2026"], errors="coerce"
        ).dt.date

    if "Дата.увольнения.2026" in df.columns:
        df["Дата.увольнения.2026"] = pd.to_datetime(
            df["Дата.увольнения.2026"], errors="coerce"
        ).dt.date

    # Для каждой строки создаем список дат работы.
    df["Дата"] = df.apply(
        lambda row: _expand_dates(row["Дата.приема.2026"], row["Дата.увольнения.2026"]),
        axis=1,
    )

    # Разворачиваем список дат в отдельные строки.
    df = df.explode("Дата", ignore_index=True)
    df = df[df["Дата"].notna()].copy()
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    # Переименовываем колонки в общий формат реестра.
    df = df.rename(
        columns={
            "Дата.приема.2026": "Дата начала работы",
            "Дата.увольнения.2026": "Дата окончания работы",
            "Источник": "План/Факт",
            "ФИО ": "ФИО",
            "Должность": "Профессия",
        }
    )

    # Оставляем только 2026 год, как в Power Query.
    df = df[pd.to_datetime(df["Дата"], errors="coerce").dt.year == 2026].copy()

    return df


In [83]:

# ------------------------------
# 2. ПЛАН / ФАКТ ПО ЧИСЛЕННОСТИ
# ------------------------------

def transform_staff_count(df: pd.DataFrame) -> pd.DataFrame:
    """
    Универсальная функция для файлов:
    - План_численность.xlsx
    - Факт_численность.xlsx

    Логика повторяет Power Query:
    - приводит типы;
    - разворачивает период [Дата начала работы] -> [Дата окончания работы]
      в ежедневные строки;
    - удаляет лишние колонки;
    - приводит названия колонок к общему формату.
    """
    df = df.copy()

    # Текстовые колонки приводим к string.
    text_cols = [
        "План/Факт",
        "Подразделение",
        "Отдел",
        "ФИО работника",
        "Должность/профессияработника",
    ]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")

    # Приводим даты к типу date.
    if "Дата начала работы" in df.columns:
        df["Дата начала работы"] = pd.to_datetime(
            df["Дата начала работы"], errors="coerce"
        ).dt.date

    if "Дата окончания работы" in df.columns:
        df["Дата окончания работы"] = pd.to_datetime(
            df["Дата окончания работы"], errors="coerce"
        ).dt.date

    # Разворачиваем период в ежедневные даты.
    df["Дата"] = df.apply(
        lambda row: _expand_dates(
            row["Дата начала работы"],
            row["Дата окончания работы"],
        ),
        axis=1,
    )

    df = df.explode("Дата", ignore_index=True)
    df = df[df["Дата"].notna()].copy()
    df["Дата"] = pd.to_datetime(df["Дата"], errors="coerce").dt.date

    # Удаляем лишние колонки.
    df.drop(
        columns=["Год", "Пол", "Основная профессия при приеме"],
        errors="ignore",
        inplace=True,
    )

    # Переименовываем колонки в общий формат.
    df = df.rename(
        columns={
            "Отдел": "ЦФО",
            "ФИО работника": "ФИО",
            "Должность/профессияработника": "Профессия",
        }
    )

    return df


In [84]:

# ------------------------------
# 3. ЕДИНЫЙ РЕЕСТР
# ------------------------------

def transform_registry(
    df_initial_plan: pd.DataFrame,
    df_plan_count: pd.DataFrame,
    df_fact_count: pd.DataFrame,
) -> pd.DataFrame:
    """
    Собирает единый реестр из трёх подготовленных источников.

    Что делает функция:
    1. Объединяет три таблицы.
    2. Оставляет только 2025 и 2026 годы.
    3. Нормализует подразделения.
    4. Добавляет служебные колонки: Год, ДеньМесяц, Месяц.
    5. Нормализует названия профессий.

    Важно:
    - на этом шаге мы НЕ считаем D, D+1, D+7;
    - горизонты считаются отдельно, чтобы не путать сырой реестр и итоговую витрину.
    """
    df = pd.concat(
        [df_initial_plan, df_plan_count, df_fact_count],
        ignore_index=True,
        sort=False,
    ).copy()

    # Приводим даты к datetime.
    df = _to_datetime_safe(df, ["Дата", "Дата начала работы", "Дата окончания работы"])

    # Оставляем только нужные годы.
    df["_ГодФильтр"] = df["Дата"].dt.year
    df = df[df["_ГодФильтр"].isin([2025, 2026])].copy()

    # Удаляем временные колонки.
    df.drop(columns=["Примечание", "_ГодФильтр"], errors="ignore", inplace=True)

    # Служебные колонки для аналитики.
    df["Год"] = df["Дата"].dt.year
    df["ДеньМесяц"] = df["Дата"].dt.strftime("%d.%m")
    df["Месяц"] = df["Дата"].dt.month.map(RU_MONTHS)

    # Нормализуем подразделения.
    if "Подразделение" in df.columns:
        df["Подразделение"] = df["Подразделение"].astype("string")
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Административно-хозяйственный отдел",
            "Администрация",
            regex=False,
        )
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Участок ",
            "",
            regex=False,
        )
        df["Подразделение"] = df["Подразделение"].str.replace(
            "Эрел",
            "Хаптагай-Хая",
            regex=False,
        )

    # Формируем КлючУчета так же, как в Power Query,
    # но потом удаляем, потому что в твоем M-коде он тоже удалялся.
    fio = (
        df["ФИО"].astype("string").fillna("")
        if "ФИО" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )
    подразделение = (
        df["Подразделение"].astype("string").fillna("")
        if "Подразделение" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )
    профессия = (
        df["Профессия"].astype("string").fillna("")
        if "Профессия" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )

    дата_начала = (
        pd.to_datetime(df["Дата начала работы"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
        if "Дата начала работы" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )
    дата_окончания = (
        pd.to_datetime(df["Дата окончания работы"], errors="coerce").dt.strftime("%Y%m%d").fillna("")
        if "Дата окончания работы" in df.columns
        else pd.Series("", index=df.index, dtype="string")
    )

    df["КлючУчета"] = np.where(
        fio == "Вакансия",
        подразделение + "|" + профессия + "|" + дата_начала + "|" + дата_окончания,
        fio,
    )

    # Нормализуем названия профессий.
    prof_replace = {
        "Электрослесарь (слесарь) дежурный и по ремонту оборудования": "Электрослесарь дежурный по ремонту оборудования",
        "Слесарь ТО": "Слесарь по техническому обслуживанию",
        "Водитель погрузчика": "Машинист погрузчика",
        "Мастер горный": "Горный мастер",
        "Электрогазосварщик, занятый на резке и ручной сварке": "Электрогазосварщик",
        "Комендант общежития": "Комендант",
        "Водитель автомобиля - экспедитор": "Водитель-экспедитор",
        "Слесарь на монтаже промывочных установок": "Слесарь-монтажник промывочного оборудования",
        "Слесарь дежурный по ремонту оборудования": "Слесарь",
        "Машинист колесного бульдозера": "Машинист погрузчика",
        "Машинист буровой установки POC": "Машинист буровой установки D55",
    }
    if "Профессия" in df.columns:
        df["Профессия"] = df["Профессия"].replace(prof_replace)

    # Удаляем временный КлючУчета, как в твоём M-коде.
    df.drop(columns=["КлючУчета"], errors="ignore", inplace=True)

    # Если есть колонка Id, приводим её к целочисленному типу.
    if "Id" in df.columns:
        df["Id"] = pd.to_numeric(df["Id"], errors="coerce").astype("Int64")

    return df


In [85]:

# ------------------------------
# 4. ВИТРИНА РЕСУРСОВ ПО ГОРИЗОНТАМ
# ------------------------------

def build_resources_horizon(df: pd.DataFrame, group_cols=None) -> pd.DataFrame:
    """
    Строит витрину ресурсов по горизонтам:
    - Ресурсы_t   : ресурсы на дату D
    - Ресурсы_d+1 : ресурсы на дату D+1, показанные в строке даты D
    - Ресурсы_d+7 : ресурсы на дату D+7, показанные в строке даты D

    Почему здесь используется '-1' и '-7':
    -------------------------------------------------------------
    Если ресурс существует на 17.04, то в отчёте на дату 16.04
    он должен попасть в колонку D+1.

    Значит, значение с даты 17.04 надо перенести на строку 16.04:
        17.04 - 1 день = 16.04

    То же самое для D+7:
        23.04 - 7 дней = 16.04

    Это именно логика отчёта по горизонту.
    """
    if group_cols is None:
        group_cols = ["Подразделение"]

    work = df.copy()
    work["Дата"] = pd.to_datetime(work["Дата"], errors="coerce").dt.normalize()

    # Берем только строки ресурсов.
    work = work[work["План/Факт"].eq("Ресурсы для выполнения программы")].copy()

    # Базовое количество ресурсов на дату D.
    t = (
        work.groupby(["Дата"] + group_cols, dropna=False)
        .size()
        .reset_index(name="Ресурсы_t")
    )

    # Ресурсы на D+1 переносим на строку даты D.
    d1 = (
        work.assign(Дата=work["Дата"] - pd.Timedelta(days=1))
        .groupby(["Дата"] + group_cols, dropna=False)
        .size()
        .reset_index(name="Ресурсы_d+1")
    )

    # Ресурсы на D+7 переносим на строку даты D.
    d7 = (
        work.assign(Дата=work["Дата"] - pd.Timedelta(days=7))
        .groupby(["Дата"] + group_cols, dropna=False)
        .size()
        .reset_index(name="Ресурсы_d+7")
    )

    # Объединяем три горизонта в одну витрину.
    result = (
        t.merge(d1, on=["Дата"] + group_cols, how="outer")
         .merge(d7, on=["Дата"] + group_cols, how="outer")
         .fillna(0)
    )

    # Приводим числовые колонки к int.
    for col in ["Ресурсы_t", "Ресурсы_d+1", "Ресурсы_d+7"]:
        result[col] = result[col].astype(int)

    # Добавляем удобные поля для сводных и фильтров.
    result["ДеньМесяц"] = result["Дата"].dt.strftime("%d.%m")
    result["Месяц"] = result["Дата"].dt.month.map(RU_MONTHS)
    result["Год"] = result["Дата"].dt.year

    return result.sort_values(["Дата"] + group_cols).reset_index(drop=True)


In [86]:

# ------------------------------
# 5. ЗАГРУЗКА ИСХОДНЫХ ДАННЫХ
# ------------------------------

# При необходимости поправь имена файлов под свою папку source.
initial_plan_path = BASE_PATH / "Первоначальный план.xlsx"
plan_count_path = BASE_PATH / "План_численность.xlsx"
fact_count_path = BASE_PATH / "Факт_численность.xlsx"


# Читаем исходные Excel-файлы.
df_initial_plan_raw = pd.read_excel(initial_plan_path, sheet_name="Лист1")
df_plan_count_raw = pd.read_excel(plan_count_path, sheet_name="Лист1")
df_fact_count_raw = pd.read_excel(fact_count_path, sheet_name="Лист1")


# Подготавливаем каждую таблицу отдельно.
df_initial_plan = transform_initial_plan(df_initial_plan_raw)
df_plan_count = transform_staff_count(df_plan_count_raw)
df_fact_count = transform_staff_count(df_fact_count_raw)


In [87]:

# ------------------------------
# 6. СБОРКА РЕЕСТРА И ВИТРИНЫ РЕСУРСОВ
# ------------------------------

# Единый реестр всех источников.
registry_df = transform_registry(
    df_initial_plan=df_initial_plan,
    df_plan_count=df_plan_count,
    df_fact_count=df_fact_count,
)

# Витрина ресурсов по подразделениям.
# resources_by_department_df = build_resources_horizon(
#     registry_df,
#     group_cols=["Подразделение"],
# )

# При необходимости можно строить более детальную витрину:
resources_by_department_prof_df = build_resources_horizon(
     registry_df,
     group_cols=["Подразделение", "Профессия", "ФИО"],
 )

print("Реестр:", registry_df.shape)
print("Ресурсы по подразделениям:", resources_by_department_df.shape)

#registry_df.head(20)


Реестр: (571803, 15)
Ресурсы по подразделениям: (4533, 8)


In [93]:

# ------------------------------
# 7. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ------------------------------

# Сохраняем результат в один Excel-файл с несколькими листами.
output_path = Path("Реестр_и_ресурсы.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    registry_df.to_excel(writer, sheet_name="Реестр", index=False)
    df.to_excel(writer, sheet_name="Ресурсы_подразделения", index=False)

print(f"Файл сохранен: {output_path.resolve()}")


Файл сохранен: C:\Users\poisk-12\PycharmProjects\Personals\Реестр_и_ресурсы.xlsx


In [88]:
def merge_resources_df_plan(df1, df2):
    df1 = df1.copy()
    df2 = df2.copy()

    df1["Дата"]=pd.to_datetime(df1["Дата"], errors="coerce")
    df2["Дата"]=pd.to_datetime(df2["Дата"], errors="coerce")


    return df1.merge(
        df2,
        on=["ФИО", "Дата", "Подразделение", "Профессия"],
        how="left",
        validate="one_to_one",
        indicator=True
    )


In [89]:
df1=resources_by_department_prof_df
df2=df_plan_count
df=merge_resources_df_plan(df1,df2)

In [90]:
df1.info()
df2.info()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 118263 entries, 0 to 118262
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Дата           118263 non-null  datetime64[us]
 1   Подразделение  118263 non-null  string        
 2   Профессия      118263 non-null  string        
 3   ФИО            118263 non-null  string        
 4   Ресурсы_t      118263 non-null  int64         
 5   Ресурсы_d+1    118263 non-null  int64         
 6   Ресурсы_d+7    118263 non-null  int64         
 7   ДеньМесяц      118263 non-null  str           
 8   Месяц          118263 non-null  str           
 9   Год            118263 non-null  int32         
dtypes: datetime64[us](1), int32(1), int64(3), str(2), string(3)
memory usage: 8.6 MB
<class 'pandas.DataFrame'>
Index: 114397 entries, 0 to 114397
Data columns (total 12 columns):
 #   Column                            Non-Null Count   Dtype         
---  ------             

In [92]:
df["План/Факт"].unique()

<StringArray>
[<NA>, 'План']
Length: 2, dtype: string

In [95]:
df

,Дата,Подразделение,Профессия,ФИО,Ресурсы_t,Ресурсы_d+1,Ресурсы_d+7,ДеньМесяц,Месяц,Год,План/Факт,ЦФО,Дата начала работы,Дата окончания работы,Дата увольнения?,Дата звонка работнику,Подтверждение приезда работником,Подтверждение принял,_merge
0,2025-12-25,Администрация,Бухгалтер по налоговому учету,Вожакова Ирина Юрьевна,0,0,1,25.12,декабрь,2025,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
1,2025-12-25,Администрация,Генеральный директор,Каздобин Андрей Витальевич,0,0,1,25.12,декабрь,2025,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
2,2025-12-25,Администрация,Главный инженер,Баранов Игорь Александрович,0,0,1,25.12,декабрь,2025,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
3,2025-12-25,Администрация,Директор по развитию минерально-сырьевой базы,Рыбаков Валерий Юрьевич,0,0,1,25.12,декабрь,2025,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
4,2025-12-25,Администрация,Заведующий отделом,Пряхина Татьяна Ивановна,0,0,1,25.12,декабрь,2025,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118258,2026-12-31,Талалах,Сторож производственных объектов,Анди Алиевич,1,0,0,31.12,декабрь,2026,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
118259,2026-12-31,Талалах,Сторож производственных объектов,Гресько Геннадий Афанасьевич,1,0,0,31.12,декабрь,2026,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
118260,2026-12-31,Талынья,Сторож производственных объектов,Вакансия,1,0,0,31.12,декабрь,2026,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
118261,2026-12-31,Юбилейный,Сторож производственных объектов,Климов Сергей Аркадьевич,1,0,0,31.12,декабрь,2026,<NA>,<NA>,NaN,NaN,NaN,NaT,NaN,NaN,left_only
